# AdaContext exploratory data analysis

This notebook loads the bundled benchmark, creates structure-aware chunks, and inspects document size, chunk size, evidence requirements, and retrieval uncertainty. Change `embedding_backend` to `sentence_transformers` for the semantic experiment.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from adacontext.config import load_config
from adacontext.datasets import load_sample_dataset
from adacontext.eda import run_eda
from adacontext.pipeline import AdaptiveContextPipeline

In [ ]:
config = load_config(overrides={'retrieval': {'embedding_backend': 'tfidf'}})
documents, questions = load_sample_dataset(ROOT)
pipeline = AdaptiveContextPipeline(config).index_documents(documents)
summary = run_eda(pipeline, questions, ROOT / 'reports' / 'generated' / 'notebook_eda')
summary

In [ ]:
import pandas as pd
chunks = pd.read_csv(ROOT / 'reports' / 'generated' / 'notebook_eda' / 'chunks.csv')
questions_df = pd.read_csv(ROOT / 'reports' / 'generated' / 'notebook_eda' / 'questions.csv')
display(chunks.groupby('document_title').agg(chunks=('chunk_id', 'count'), tokens=('token_count', 'sum')))
display(questions_df.groupby('difficulty').agg(questions=('question_id', 'count'), mean_entropy=('score_entropy', 'mean'), mean_evidence=('gold_evidence_chunks', 'mean')))

In [ ]:
import plotly.express as px
px.scatter(questions_df, x='score_entropy', y='gold_evidence_chunks', color='difficulty', hover_data=['question_id'], title='Retrieval uncertainty versus required evidence')